In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


def create_visualizer():
    """Return a ModelVisualizer with the P2 experimental notice acknowledged.

    kailash-ml 2.2.2 emits ExperimentalWarning (a UserWarning) at
    ModelVisualizer construction; exercises run under warnings-as-errors,
    so the notice is acknowledged narrowly here — the ONE construction site
    every course file should use.
    """
    import warnings as _warnings

    from kailash_ml import ModelVisualizer
    from kailash_ml._decorators import ExperimentalWarning as _EW

    with _warnings.catch_warnings():
        _warnings.simplefilter("ignore", _EW)
        return ModelVisualizer()

# ── SDK-internal deprecation nag filter (strict-gate hygiene) ────────────
# torch 2.12's ONNX exporter warns that dynamic_axes is "not recommended"
# under dynamo=True — a deprecation notice inside OnnxBridge's internals,
# not a correctness signal, and fatal under warnings-as-errors. Acknowledged
# here (kailash_helpers is imported by every shared.mlfpNN package, so every
# technique file gets it); message-matched, everything else still errors.
import warnings as _warnings

_warnings.filterwarnings(
    "ignore", message=r".*dynamic_axes.*not recommended.*", category=UserWarning
)
# matplotlib's Agg backend (forced in headless/script runs) nags
# "FigureCanvasAgg is non-interactive, and thus cannot be shown" on every
# plt.show() — a backend notice, not a correctness signal.
_warnings.filterwarnings(
    "ignore", message=r".*non-interactive.*cannot be shown.*", category=UserWarning
)



# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                module_dir = self._local_data / module
                local_path = module_dir / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
                # Bundled data is present for this module but the file is absent
                # → it is genuinely missing. Raise immediately instead of falling
                # back to a Drive download that would hang offline (and pulls the
                # whole folder online just to discover the file is not there).
                if module_dir.is_dir():
                    raise FileNotFoundError(
                        f"File not found: {module}/{filename} "
                        f"(not in local data/{module}/)."
                    )
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp03/ex_2.py ──
"""
Shared infrastructure for MLFP03 Exercise 2 — Regularisation and
Cross-Validation.

Contains: data loading for the Singapore credit scoring dataset,
feature preparation, synthetic 1D bias-variance problem, alpha grids,
plotting helpers, and a shared OUTPUT_DIR for generated artefacts.

Technique-specific code (Ridge/Lasso model construction, nested CV
loops, learning curves) lives in the per-technique solution files —
this module holds only the helpers those files share.
"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from sklearn.base import clone
from sklearn.linear_model import LinearRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# CONSTANTS
# ════════════════════════════════════════════════════════════════════════

# Deterministic seed for every random operation in this exercise
SEED = 42

# Alpha sweep used by Ridge and the Ridge regularisation path.
ALPHAS: list[float] = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]

# Lasso / ElasticNet need their OWN grid: sklearn's Lasso divides the
# squared-error term by 2n, so its α lives on a much smaller scale than
# Ridge's. With a standardised target, α ≥ ~0.5 already zeroes every
# coefficient; the grid below walks the full path from "almost OLS" to
# "intercept only".
LASSO_ALPHAS: list[float] = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0]

# Regression target for the credit demos. ``savings_balance`` is only
# partly predictable from the other columns (R² ≈ 0.25) and no single
# feature duplicates it — so regularisation has real work to do. (Contrast
# ``credit_utilization``: it is 0.97-correlated with
# ``avg_balance_utilization``, so every model would score the same and no
# regularisation effect would be visible.)
CREDIT_TARGET = "savings_balance"

# Columns that are never features in this exercise:
#   customer_id              — row identifier
#   default                  — the loan OUTCOME, observed after the fact
#   future_default_indicator — post-outcome copy of ``default`` (a planted
#                              leak; see Lesson 3.1)
CREDIT_NON_FEATURE_COLUMNS: tuple[str, ...] = (
    "customer_id",
    "default",
    "future_default_indicator",
)

# A SMALL training sample is deliberate: with ~33 correlated features and
# a few hundred rows, OLS visibly overfits (train R² well above test R²),
# which is exactly the regime where Ridge and Lasso earn their keep. The
# large test set keeps every test score stable to ~±0.01.
N_TRAIN_DEFAULT = 300
N_TEST_DEFAULT = 5000

# Output directory for HTML plots, summary CSVs, etc.
OUTPUT_DIR = Path("outputs") / "mlfp03_ex2_regularisation_cv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore credit scoring
# ════════════════════════════════════════════════════════════════════════


def load_credit_data(
    n_train: int = N_TRAIN_DEFAULT,
    n_test: int = N_TEST_DEFAULT,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, list[str]]:
    """Load + preprocess the MLFP02 Singapore credit scoring parquet.

    Returns:
        X_train, y_train, X_test, y_test, feature_names

    Target: ``savings_balance`` (S$), STANDARDISED with the training mean
    and standard deviation — so an MSE of 1.0 means "no better than
    predicting the mean" and MSE = 1 − R² (approximately) on the test set.

    ``n_train`` rows are used for fitting (small on purpose — see
    ``N_TRAIN_DEFAULT``) and ``n_test`` held-out rows for evaluation.
    ID, outcome and leak columns (``CREDIT_NON_FEATURE_COLUMNS``) are
    dropped before preprocessing.

    Uses ``kailash_ml.PreprocessingPipeline`` for normalisation +
    ordinal encoding + median imputation, fitted on the ``n_train``
    training rows only — the test rows are held out FIRST, so none of
    their statistics reach the scaler or the imputer. All regularised
    models REQUIRE normalised features (otherwise the penalty is unevenly
    distributed across the coefficient vector).
    """
    loader = MLFPDataLoader()
    credit = loader.load("mlfp02", "sg_credit_scoring.parquet").drop(
        CREDIT_NON_FEATURE_COLUMNS
    )
    credit = credit.sample(n=n_train + n_test, seed=SEED)

    # Split FIRST (exactly n_test held-out rows), then fit the preprocessing
    # on the training rows only.
    result = split_then_preprocess(
        credit,
        target=CREDIT_TARGET,
        test_size=n_test,
        seed=SEED,
        normalize=True,
        categorical_encoding="ordinal",
        imputation_strategy="median",
    )

    feature_cols = [c for c in result.train_data.columns if c != CREDIT_TARGET]
    X_train, y_train, col_info = to_sklearn_input(
        result.train_data,
        feature_columns=feature_cols,
        target_column=CREDIT_TARGET,
    )
    X_test, y_test, _ = to_sklearn_input(
        result.test_data,
        feature_columns=feature_cols,
        target_column=CREDIT_TARGET,
    )

    # Standardise the target with TRAINING statistics only (no test leakage).
    y_mean, y_std = float(y_train.mean()), float(y_train.std())
    y_train = (y_train - y_mean) / y_std
    y_test = (y_test - y_mean) / y_std

    return X_train, y_train, X_test, y_test, col_info["feature_columns"]


# Preprocessing for the credit-default CV demo. It is fitted INSIDE every
# fold (see ``cross_val_auc_split_first``), never once on the whole sample.
CREDIT_DEFAULT_PREPROCESSING: dict[str, Any] = {
    "normalize": True,
    "categorical_encoding": "ordinal",
    "imputation_strategy": "median",
}


def load_credit_default_sample(
    n: int = 600,
) -> tuple[pl.DataFrame, np.ndarray, list[str]]:
    """A small credit sample with the BINARY ``default`` outcome as target.

    Used to show why stratified k-fold matters for an imbalanced target
    (~13% defaults): with plain k-fold the default rate drifts from fold
    to fold. Returns (X_raw, y, feature_names). The ID / post-outcome leak
    columns are dropped, but ``X_raw`` is NOT preprocessed: in
    cross-validation every fold is its own train/test split, so imputation
    and scaling must be fitted inside each fold on that fold's training
    rows — fitting them once on all ``n`` rows would leak every test fold
    into its training fold. Score models with ``cross_val_auc_split_first``.
    """
    loader = MLFPDataLoader()
    credit = (
        loader.load("mlfp02", "sg_credit_scoring.parquet")
        .drop(["customer_id", "future_default_indicator"])
        .sample(n=n, seed=SEED)
    )
    y = credit["default"].to_numpy().astype(int)
    X_raw = credit.drop("default")
    return X_raw, y, X_raw.columns


def cross_val_auc_split_first(
    model: Any, X_raw: pl.DataFrame, y: np.ndarray, *, cv: Any
) -> np.ndarray:
    """ROC-AUC per fold with the preprocessing re-fitted inside every fold.

    Plays the role of ``cross_val_score(model, X, y, cv=cv,
    scoring="roc_auc")`` for RAW features: in each fold the
    ``PreprocessingPipeline`` is fitted on the training rows only and
    applied to the held-out rows, then a fresh clone of ``model`` is fitted
    and scored. Returns one AUC per fold.
    """
    frame = X_raw.with_columns(pl.Series("default", y))
    scores: list[float] = []
    for tr_idx, te_idx in cv.split(np.zeros(len(y)), y):
        prep = preprocess_train_test(
            frame[tr_idx.tolist()],
            frame[te_idx.tolist()],
            "default",
            seed=SEED,
            **CREDIT_DEFAULT_PREPROCESSING,
        )
        features = [c for c in prep.train_data.columns if c != "default"]
        X_tr, y_tr, _ = to_sklearn_input(
            prep.train_data, feature_columns=features, target_column="default"
        )
        X_te, y_te, _ = to_sklearn_input(
            prep.test_data, feature_columns=features, target_column="default"
        )
        fitted = clone(model).fit(X_tr, y_tr.astype(int))
        scores.append(float(roc_auc_score(y_te.astype(int), fitted.predict_proba(X_te)[:, 1])))
    return np.array(scores)


ICU_CV_FEATURES: list[str] = [
    "age",
    "gender",
    "height_cm",
    "weight_kg",
    "bmi",
    "insurance",
    "ethnicity",
    "diagnosis",
    "icu_type",
]


def load_icu_admissions_for_cv() -> dict[str, Any]:
    """ICU admissions with REAL time order and REAL repeated-patient groups.

    One row per admission (8,000 admissions of ~4,000 patients), sorted by
    ``admit_time`` so that TimeSeriesSplit's "earlier rows train, later
    rows test" really means "past predicts future". Features are only
    things known AT admission (patient demographics, diagnosis, ICU type);
    the target is ``los_days`` (length of stay).

    Returns a dict with X, y, groups (patient_id per row), admit_time
    (polars Series), feature_names.
    """
    loader = MLFPDataLoader()
    admissions = loader.load("mlfp02", "icu_admissions.parquet")
    patients = loader.load("mlfp02", "icu_patients.parquet")
    frame = (
        admissions.join(patients, on="patient_id", how="left")
        .with_columns(
            pl.col("admit_time").str.to_datetime("%Y-%m-%d %H:%M:%S")
        )
        .sort("admit_time")
    )
    encoded = frame.with_columns(
        [
            pl.col(c).rank("dense").cast(pl.Float64)
            for c in ICU_CV_FEATURES
            if frame.schema[c] == pl.String
        ]
    ).with_columns(
        [pl.col(c).cast(pl.Float64).fill_null(pl.col(c).median()) for c in ICU_CV_FEATURES]
    )
    return {
        "X": encoded.select(ICU_CV_FEATURES).to_numpy(),
        "y": encoded["los_days"].to_numpy(),
        "groups": encoded["patient_id"].to_numpy(),
        "admit_time": encoded["admit_time"],
        "feature_names": list(ICU_CV_FEATURES),
    }


# ════════════════════════════════════════════════════════════════════════
# SYNTHETIC 1D PROBLEM — for bias/variance and polynomial fits
# ════════════════════════════════════════════════════════════════════════


def sine_truth(x: np.ndarray) -> np.ndarray:
    """The noiseless generating function f(x) = sin(2πx)."""
    return np.sin(2 * np.pi * np.asarray(x).ravel())


def sample_sine_training_set(
    n: int,
    noise_sigma: float,
    rng: np.random.Generator,
) -> tuple[np.ndarray, np.ndarray]:
    """Draw ONE fresh training set (x ~ U[0,1], y = sin(2πx) + ε).

    The bias-variance decomposition is defined over many independent
    training sets drawn from the same process — this is how we draw them.
    """
    x = rng.uniform(0, 1, n)
    y = sine_truth(x) + rng.normal(0, noise_sigma, n)
    return x.reshape(-1, 1), y


def make_sine_dataset(
    n_train: int = 40,
    n_test: int = 1000,
    noise_sigma: float = 0.2,
    seed: int = SEED,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, float]:
    """Generate a 1D noisy-sine regression problem.

    Returns:
        x_train_2d (n_train, 1), y_train (n_train,),
        x_test_2d (n_test, 1),   y_test  (n_test,),
        noise_variance (float, σ² — the irreducible error floor)

    The true function is ``y = sin(2πx) + ε`` with ε ~ N(0, σ²).
    A small training set (40 points) makes overfitting visible; a large,
    independently drawn test set (1,000 points) makes the test MSE a
    stable estimate rather than the noise of a handful of points.
    The noise variance is returned so callers can use it in the
    bias-variance decomposition (σ² is the "irreducible noise" term).
    """
    rng = np.random.default_rng(seed)
    x_train, y_train = sample_sine_training_set(n_train, noise_sigma, rng)
    x_test, y_test = sample_sine_training_set(n_test, noise_sigma, rng)
    return x_train, y_train, x_test, y_test, noise_sigma**2


def make_poly_pipeline(degree: int) -> Pipeline:
    """Polynomial-features + scaler + linear-regression pipeline."""
    return Pipeline(
        [
            ("poly", PolynomialFeatures(degree, include_bias=False)),
            ("scaler", StandardScaler()),
            ("lr", LinearRegression()),
        ]
    )


# ════════════════════════════════════════════════════════════════════════
# REPORTING HELPERS
# ════════════════════════════════════════════════════════════════════════


def print_header(title: str) -> None:
    """Print a banner so each phase of a technique file is easy to spot."""
    print("\n" + "=" * 72)
    print(f"  {title}")
    print("=" * 72)


def format_results_table(rows: list[dict[str, Any]], cols: list[str]) -> str:
    """Render a small table of dicts as fixed-width text."""
    header = "  ".join(f"{c:>12}" for c in cols)
    sep = "-" * len(header)
    body = "\n".join(
        "  ".join(
            f"{row[c]:>12.4f}" if isinstance(row[c], float) else f"{row[c]:>12}"
            for c in cols
        )
        for row in rows
    )
    return f"{header}\n{sep}\n{body}"


def save_html_plot(fig: Any, name: str) -> Path:
    """Write a plotly figure into OUTPUT_DIR and return the path."""
    path = OUTPUT_DIR / name
    fig.write_html(str(path))
    return path




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 2.4: Cross-Validation Strategies
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Run nested CV and measure the optimism of "tune and report on the
#     same folds"
#   - Use stratified k-fold to keep a rare outcome's rate equal across folds
#   - Apply TimeSeriesSplit for walk-forward validation on time-ordered data
#   - Use GroupKFold so repeat patients never sit on both sides of a split
#   - Pick the RIGHT CV strategy for a given deployment scenario
#
# PREREQUISITES:
#   - 02_ridge_regression.py and 03_lasso_elasticnet.py
#   - MLFP02 sampling theory (bias, variance of estimators)
#
# ESTIMATED TIME: ~45 minutes
#
# TASKS (5-phase R10):
#   1. Theory — why "one CV fits all" is wrong
#   2. Build — the splitters and the datasets whose structure they match
#   3. Train — nested CV (honest estimate after α selection)
#   4. Visualise — stratified, walk-forward and grouped CV on real structure
#   5. Apply — choosing the CV for a payments-fraud scorer
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import (
    GridSearchCV,
    GroupKFold,
    KFold,
    StratifiedKFold,
    TimeSeriesSplit,
    cross_val_score,
)



## THEORY — Why Cross-Validation Is More Than Just k-fold


In [ ]:
# Standard k-fold implicitly assumes:
#   (a) Observations are INDEPENDENT — shuffling is harmless
#   (b) Observations are IDENTICALLY distributed — training-time
#       distribution equals prediction-time distribution
#   (c) Hyperparameter selection and performance estimation can share
#       the same splits (they can't — the chosen setting was picked
#       BECAUSE it looked good on those folds)
#
# Real datasets break these assumptions all the time:
#   - FINANCIAL / CLINICAL DATA is temporal: shuffling lets the model
#     train on later records to predict earlier ones (violates (b)).
#   - MEDICAL DATA has repeated admissions per patient: the same patient
#     can end up in both train and test (violates (a)).
#   - RARE OUTCOMES (defaults, fraud) make a random fold's positive rate
#     swing, so fold scores are noisier than they need to be.
#   - HYPERPARAMETER TUNING on the evaluation folds gives an optimistic
#     estimate (violates (c)).
#
# THE FIX — pick the CV strategy that MATCHES the deployment scenario:
#   i.i.d. data             → k-fold
#   rare binary outcome     → stratified k-fold
#   temporal data           → TimeSeriesSplit (walk-forward)
#   grouped data            → GroupKFold
#   hyperparameter tuning   → nested CV (outer for eval, inner for tune)



## TASK 2 — BUILD: the datasets and their structure


In [ ]:
# Each strategy is demonstrated on data that ACTUALLY has the structure
# it is designed for:
#   - credit savings regression (300 rows)   → nested CV
#   - credit default, 600 applicants (~13%)  → stratified k-fold
#   - ICU admissions, time-ordered, with
#     repeat patients                        → TimeSeriesSplit, GroupKFold

print_header("Cross-Validation Strategies")
X_train, y_train, X_test, y_test, feature_names = load_credit_data()
print(f"Credit regression train: {X_train.shape}")

X_def, y_def, _ = load_credit_default_sample(n=600)
print(f"Credit default sample:   {X_def.shape}, default rate {y_def.mean():.1%}")

icu = load_icu_admissions_for_cv()
n_patients = len(np.unique(icu["groups"]))
print(
    f"ICU admissions:          {icu['X'].shape}, {n_patients} distinct patients, "
    f"{icu['admit_time'].min():%Y-%m-%d} → {icu['admit_time'].max():%Y-%m-%d}"
)



## TASK 3 — TRAIN with nested CV (honest estimate after α selection)


In [ ]:
# STANDARD CV uses the SAME folds to pick α AND report performance: the
# best mean CV score among the candidates. That number is optimistic —
# among several candidates, the winner is partly the one that got lucky
# on those particular folds.
#
# NESTED CV fixes this:
#   OUTER 5-fold: held out for performance reporting (never touched
#                 during α selection).
#   INNER 3-fold: used inside each outer fold to pick α from ALPHAS.
#
# The outer mean estimates how well the WHOLE procedure ("tune α by CV,
# then fit") generalises.

print_header("Nested Cross-Validation (Ridge on credit savings)")

# Standard (optimistic) CV: best mean CV score across the α grid
grid = GridSearchCV(
    Ridge(),
    {"alpha": ALPHAS},
    cv=KFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring="r2",
)
grid.fit(X_train, y_train)
# TODO: The standard (optimistic) score is the BEST mean CV score found by
# the grid search. Hint: a GridSearchCV attribute ending in an underscore.
standard_score = ____
print(
    f"Standard CV (tune + report on same folds): R² = {standard_score:.4f}, "
    f"selected α = {grid.best_params_['alpha']}"
)

outer_cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
inner_cv = KFold(n_splits=3, shuffle=True, random_state=SEED)

nested_scores: list[float] = []
selected_alphas: list[float] = []
print("\nOuter folds:")
for fold_idx, (tr_idx, te_idx) in enumerate(outer_cv.split(X_train)):
    X_out_tr, X_out_te = X_train[tr_idx], X_train[te_idx]
    y_out_tr, y_out_te = y_train[tr_idx], y_train[te_idx]

    best_alpha = ALPHAS[0]
    best_inner = -np.inf
    for alpha in ALPHAS:
        # TODO: inner 3-fold CV of Ridge(alpha=alpha) on the OUTER-TRAIN rows
        # only (X_out_tr, y_out_tr), cv=inner_cv, scoring="r2".
        inner = ____
        if inner.mean() > best_inner:
            best_inner = inner.mean()
            best_alpha = alpha

    # TODO: refit Ridge at best_alpha on the outer-train rows and score R²
    # on the outer-test rows.
    ridge_selected = ____
    outer_score = ____
    nested_scores.append(outer_score)
    selected_alphas.append(best_alpha)
    print(
        f"  Fold {fold_idx + 1}: α = {best_alpha:<8.4f}  "
        f"outer R² = {outer_score:.4f}"
    )

nested_mean = float(np.mean(nested_scores))
nested_std = float(np.std(nested_scores))
optimism = standard_score - nested_mean
print(f"\nNested CV:   R² = {nested_mean:.4f} ± {nested_std:.4f}")
print(f"Standard CV: R² = {standard_score:.4f}")
print(f"Optimism (standard - nested): {optimism:+.4f}")



### Checkpoint 1


In [ ]:
assert len(nested_scores) == 5, "Should have 5 outer fold scores"
assert all(isinstance(s, float) for s in nested_scores), "Scores must be floats"
print("\n[ok] Checkpoint 1 passed — nested CV estimate produced")
print(
    f"  The optimism here is {optimism:+.4f} R², against a fold-to-fold "
    f"spread of ±{nested_std:.4f}. "
    + (
        "It is larger than one standard deviation — report the nested number."
        if optimism > nested_std
        else "It is within the fold noise on this small sample, but the "
        "nested number is still the one to report."
    )
)



## TASK 4 — VISUALISE stratified, walk-forward and grouped CV


### 4a. Stratified k-fold on a rare outcome


default rate per fold          AUC (mean ± sd)
  KFold              {min(plain_rates):.3f} – {max(plain_rates):.3f}                 {plain_auc.mean():.3f} ± {plain_auc.std():.3f}
  StratifiedKFold    {min(strat_rates):.3f} – {max(strat_rates):.3f}                 {strat_auc.mean():.3f} ± {strat_auc.std():.3f}


In [ ]:
print_header("Stratified vs plain k-fold — credit default (rare outcome)")

plain_cv = KFold(n_splits=10, shuffle=True, random_state=SEED)
# TODO: a 10-fold StratifiedKFold (shuffle=True, random_state=SEED)
strat_cv = ____
plain_rates = [float(y_def[te].mean()) for _, te in plain_cv.split(X_def, y_def)]
# TODO: default rate of each stratified test fold (mirror plain_rates).
strat_rates = ____
logit = LogisticRegression(max_iter=2000)
# X_def is RAW: each fold re-fits the imputer and scaler on its own
# training rows. Fitting them once on all 600 rows would let every test
# fold's statistics into its training fold.
plain_auc = cross_val_auc_split_first(logit, X_def, y_def, cv=plain_cv)
strat_auc = cross_val_auc_split_first(logit, X_def, y_def, cv=strat_cv)
print(
)

fig_strat = go.Figure()
fig_strat.add_trace(go.Bar(x=list(range(1, 11)), y=plain_rates, name="KFold"))
fig_strat.add_trace(go.Bar(x=list(range(1, 11)), y=strat_rates, name="StratifiedKFold"))
fig_strat.add_hline(y=float(y_def.mean()), line_dash="dot", annotation_text="overall rate")
fig_strat.update_layout(
    title="Default rate in each test fold",
    xaxis_title="Fold",
    yaxis_title="Default rate",
    barmode="group",
)
print(f"Saved: {save_html_plot(fig_strat, 'ex2_04_stratified_folds.html')}")



### 4b. Walk-forward (TimeSeriesSplit) on time-ordered ICU admissions


In [ ]:
print_header("TimeSeriesSplit + GroupKFold — ICU length of stay")
X_icu, y_icu, groups = icu["X"], icu["y"], icu["groups"]
admit_time = icu["admit_time"]

# TODO: a 5-split walk-forward splitter
tscv = ____
print("\nWalk-forward splits (rows are sorted by admit_time):")
for fold, (tr_idx, te_idx) in enumerate(tscv.split(X_icu)):
    assert admit_time[int(tr_idx[-1])] <= admit_time[int(te_idx[0])]
    print(
        f"  Fold {fold + 1}: train {admit_time[0]:%Y-%m} → "
        f"{admit_time[int(tr_idx[-1])]:%Y-%m} ({len(tr_idx)} adm.), "
        f"test {admit_time[int(te_idx[0])]:%Y-%m} → "
        f"{admit_time[int(te_idx[-1])]:%Y-%m} ({len(te_idx)} adm.)"
    )



### 4c. GroupKFold: same patient never in both train and test


In [ ]:
shuffled_kfold = KFold(n_splits=5, shuffle=True, random_state=SEED)
group_cv = GroupKFold(n_splits=5)
leaky_patients = []
for (tr_k, te_k), (tr_g, te_g) in zip(
    shuffled_kfold.split(X_icu), group_cv.split(X_icu, groups=groups)
):
    # TODO: count patients present in BOTH the KFold train and test rows.
    leaky_patients.append(____)
    assert not (set(groups[tr_g]) & set(groups[te_g])), "GroupKFold leaked a patient"
print(
    f"\nShuffled KFold: on average {np.mean(leaky_patients):.0f} patients per fold "
    "appear in BOTH train and test."
)
print("GroupKFold:     0 patients shared between train and test (verified).")

model = Ridge(alpha=1.0)
icu_scores = {
    "KFold (shuffled)": cross_val_score(model, X_icu, y_icu, cv=shuffled_kfold, scoring="r2"),
    "TimeSeriesSplit": cross_val_score(model, X_icu, y_icu, cv=tscv, scoring="r2"),
    # TODO: GroupKFold needs the patient ids passed as groups=
    "GroupKFold": ____,
}
print(f"\n{'Strategy':<18} {'R² (mean ± sd)':>18}")
print("-" * 38)
for name, sc in icu_scores.items():
    print(f"{name:<18} {sc.mean():>+9.4f} ± {sc.std():.4f}")

fig_cv = go.Figure()
for name, sc in icu_scores.items():
    fig_cv.add_trace(go.Box(y=sc, name=name, boxpoints="all"))
fig_cv.update_layout(
    title="ICU length-of-stay R² per fold under three CV strategies",
    yaxis_title="R² on the held-out fold",
)
print(f"Saved: {save_html_plot(fig_cv, 'ex2_04_cv_strategies.html')}")



### Checkpoint 2


In [ ]:
assert all(len(sc) == 5 for sc in icu_scores.values()), "Each strategy: 5 scores"
assert max(strat_rates) - min(strat_rates) <= max(plain_rates) - min(plain_rates), (
    "Stratified folds should keep the default rate at least as even as KFold"
)
assert np.mean(leaky_patients) > 0, "Shuffled KFold should split some patients"
print("[ok] Checkpoint 2 passed — strategies match the data structure")

best_icu = max(sc.mean() for sc in icu_scores.values())
print(
    "  "
    + (
        f"Every strategy gives R² ≤ {best_icu:.3f}: in this dataset, length of "
        "stay is essentially unpredictable from admission-time demographics, "
        "so the strategies cannot disagree much. That is itself a finding — "
        "and the honest walk-forward / grouped estimates are the ones you "
        "would report."
        if best_icu < 0.05
        else "Compare the means: a k-fold score clearly above the walk-forward "
        "or grouped score means shuffled CV was leaking time or patients."
    )
)



## TASK 5 — APPLY: choosing the CV for a payments-fraud scorer


Data property                     | CV strategy
----------------------------------|----------------------------------
Scores tomorrow's transactions    | TimeSeriesSplit (walk-forward)
Many rows per merchant / user     | GroupKFold (group = merchant/user)
Fraud is rare                     | Stratified folds (or stratified
                                  |   time blocks)
Threshold / hyperparameters tuned | Nested CV, or a final untouched
                                  |   time-ordered holdout


In [ ]:
# SCENARIO (illustrative): a Southeast Asian digital-wallet operator
# builds a real-time fraud scorer that sees each transaction once. The
# data has STRONG structure:
#   - Fraud patterns shift every few weeks (new attack vectors)
#   - Merchant mix changes with marketing campaigns
#   - Seasonal peaks (11.11, year-end sales, Chinese New Year) create
#     regime shifts that a shuffled k-fold mixes together
#   - Large merchants and repeat users contribute thousands of rows each
#
# WHY TIME-SERIES CV: walk-forward validation mirrors deployment — train
# on the past, score the future. If walk-forward performance degrades
# across successive folds, the model is ageing and needs refreshing.
#
# WHY GROUPKFOLD TOO: if one merchant (or user) sits in both train and
# test, the model can memorise that merchant and look better than it
# will on NEW merchants. Grouping by merchant forces the estimate to
# reflect generalisation to unseen merchants.
#
# WHY STRATIFIED + NESTED: fraud is rare, so fold-level fraud rates must
# be held steady; and every threshold or hyperparameter tuned on the
# evaluation folds inflates the reported recall — nested CV removes that.
#
# THE RISK OF GETTING IT WRONG (illustrative): a team that validates with
# shuffled k-fold can report a recall that the live system never
# reaches, because the validation folds contained the same time period,
# merchants and users as the training folds. The gap shows up only after
# deployment, as fraud losses.

print_header("Payments Fraud — matching CV to deployment")
print(
)



## REFLECTION


======================================================================
  WHAT YOU'VE MASTERED
======================================================================

  [x] Nested CV: outer for honest eval, inner for α selection
  [x] Measuring optimism as (best standard-CV score - nested score)
  [x] Stratified k-fold: equal outcome rates in every fold
  [x] TimeSeriesSplit: walk-forward validation, no future leakage
  [x] GroupKFold: repeat patients stay on one side of the split
  [x] Picking a CV strategy based on DEPLOYMENT, not data shape

  KEY INSIGHT: The CV strategy is a MODELLING DECISION, not a
  technicality. The "best" model under shuffled k-fold can be the
  worst model in production if the data has structure.

  NEXT: 05_learning_curves.py — learning curves diagnose "do I need
  more data or a better model?" and tie the whole exercise together.


In [ ]:
print(
)

